# Data Check - HIVAU-70k-NEW

In [31]:
import json

with open("ucf_database_old.json") as f:
    old_data = json.load(f)

with open("ucf_database_train.json") as f:
    train_data = json.load(f)

print(f"old_data: {len(old_data)} videos")
print(f"train_data: {len(train_data)} videos")

old_data: 1493 videos
train_data: 1545 videos


## 1. Tong so video anomaly

In [32]:
# Total anomaly videos in train (label != [])
all_anomaly_videos = {
    k: v for k, v in train_data.items()
    if v.get("label", []) != []
}
print(f"Tong so video anomaly trong train: {len(all_anomaly_videos)}")

# Videos can kiem tra: co trong ca old (co events_summary) va train
anomaly_videos = {
    k: v for k, v in all_anomaly_videos.items()
    if k in old_data
}
missing_in_old = set(all_anomaly_videos.keys()) - set(old_data.keys())
print(f"So video thuc su can kiem tra (co trong old, co events_summary): {len(anomaly_videos)}")
print(f"So video thieu trong old (khong co events_summary): {len(missing_in_old)}")

Tong so video anomaly trong train: 810
So video thuc su can kiem tra (co trong old, co events_summary): 758
So video thieu trong old (khong co events_summary): 52


## 2. Video co events >= 2 va co summary "no anomaly"

In [33]:
# Videos with events >= 2 and at least one events_summary contains "no anomaly"
multi_event_with_no_anomaly = {}

for k, v in anomaly_videos.items():
    events = v.get("events", [])
    if len(events) < 2:
        continue

    old_entry = old_data[k]
    summaries = old_entry.get("events_summary", [])

    no_anomaly_indices = []
    for i, s in enumerate(summaries):
        if "no anomaly" in s.lower():
            no_anomaly_indices.append(i)

    if no_anomaly_indices:
        multi_event_with_no_anomaly[k] = {
            "num_events": len(events),
            "num_summaries": len(summaries),
            "no_anomaly_indices": no_anomaly_indices,
            "total_no_anomaly": len(no_anomaly_indices),
        }

print(f"So video co events >= 2 va co it nhat 1 summary 'no anomaly': {len(multi_event_with_no_anomaly)}")
print()
for k, info in list(multi_event_with_no_anomaly.items())[:10]:
    print(f"  {k}: {info['num_events']} events, no_anomaly tai index {info['no_anomaly_indices']}")

So video co events >= 2 va co it nhat 1 summary 'no anomaly': 97

  Abuse001_x264: 2 events, no_anomaly tai index [1]
  Abuse003_x264: 3 events, no_anomaly tai index [1]
  Abuse007_x264: 3 events, no_anomaly tai index [1]
  Abuse010_x264: 2 events, no_anomaly tai index [1]
  Abuse013_x264: 2 events, no_anomaly tai index [0]
  Abuse019_x264: 4 events, no_anomaly tai index [0]
  Abuse022_x264: 4 events, no_anomaly tai index [1, 3]
  Abuse024_x264: 2 events, no_anomaly tai index [0]
  Abuse031_x264: 2 events, no_anomaly tai index [0, 1]
  Abuse037_x264: 2 events, no_anomaly tai index [1]


## 2.1. Video ma TOAN BO summary deu la "no anomaly"

In [34]:
# Videos where ALL summaries are "no anomaly"
all_no_anomaly = {
    k: info for k, info in multi_event_with_no_anomaly.items()
    if info["total_no_anomaly"] == info["num_summaries"]
}

# Videos where only SOME summaries are "no anomaly"
partial_no_anomaly = {
    k: info for k, info in multi_event_with_no_anomaly.items()
    if info["total_no_anomaly"] < info["num_summaries"]
}

print(f"Video TOAN BO summary la 'no anomaly': {len(all_no_anomaly)}")
print(f"Video CHI MOT SO summary la 'no anomaly': {len(partial_no_anomaly)}")
print()

print("=== TOAN BO no anomaly ===")
for k, info in list(all_no_anomaly.items())[:10]:
    print(f"  {k}: {info['num_events']} events, {info['num_summaries']} summaries - ALL no anomaly")

print()
print("=== MOT SO no anomaly ===")
for k, info in list(partial_no_anomaly.items())[:10]:
    print(f"  {k}: {info['num_events']} events, no_anomaly tai index {info['no_anomaly_indices']} / {info['num_summaries']} summaries")

Video TOAN BO summary la 'no anomaly': 16
Video CHI MOT SO summary la 'no anomaly': 81

=== TOAN BO no anomaly ===
  Abuse024_x264: 2 events, 1 summaries - ALL no anomaly
  Abuse031_x264: 2 events, 2 summaries - ALL no anomaly
  Abuse038_x264: 2 events, 2 summaries - ALL no anomaly
  Abuse041_x264: 2 events, 2 summaries - ALL no anomaly
  Arson034_x264: 2 events, 2 summaries - ALL no anomaly
  Fighting041_x264: 2 events, 2 summaries - ALL no anomaly
  RoadAccidents033_x264: 2 events, 2 summaries - ALL no anomaly
  RoadAccidents118_x264: 3 events, 3 summaries - ALL no anomaly
  Shoplifting014_x264: 4 events, 4 summaries - ALL no anomaly
  Stealing011_x264: 2 events, 2 summaries - ALL no anomaly

=== MOT SO no anomaly ===
  Abuse001_x264: 2 events, no_anomaly tai index [1] / 2 summaries
  Abuse003_x264: 3 events, no_anomaly tai index [1] / 3 summaries
  Abuse007_x264: 3 events, no_anomaly tai index [1] / 3 summaries
  Abuse010_x264: 2 events, no_anomaly tai index [1] / 2 summaries
  Abus

## 2.2. Ti le events "no anomaly" tren tong so events

In [35]:
# Dem tong so events va so events co summary "no anomaly"
total_events = 0
no_anomaly_events = 0

for k, v in anomaly_videos.items():
    events = v.get("events", [])
    total_events += len(events)

    old_entry = old_data[k]
    summaries = old_entry.get("events_summary", [])
    for s in summaries:
        if "no anomaly" in s.lower():
            no_anomaly_events += 1

print(f"Tong so events: {total_events}")
print(f"So events co summary 'no anomaly': {no_anomaly_events}")
print(f"Ti le: {no_anomaly_events}/{total_events} = {no_anomaly_events/total_events*100:.2f}%")

Tong so events: 1511
So events co summary 'no anomaly': 151
Ti le: 151/1511 = 9.99%


## Summary

In [36]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"1. Tong video anomaly trong train:              {len(all_anomaly_videos)}")
print(f"   - Co events_summary (can kiem tra):          {len(anomaly_videos)}")
print(f"   - Thieu trong old (khong kiem tra):          {len(missing_in_old)}")
print(f"2. Events >= 2, co 'no anomaly' summary:        {len(multi_event_with_no_anomaly)}")
print(f"   2.1. TOAN BO summary la 'no anomaly':        {len(all_no_anomaly)}")
print(f"   2.2. CHI MOT SO summary la 'no anomaly':     {len(partial_no_anomaly)}")
print("=" * 60)

SUMMARY
1. Tong video anomaly trong train:              810
   - Co events_summary (can kiem tra):          758
   - Thieu trong old (khong kiem tra):          52
2. Events >= 2, co 'no anomaly' summary:        97
   2.1. TOAN BO summary la 'no anomaly':        16
   2.2. CHI MOT SO summary la 'no anomaly':     81


## 3. Tao ucf_database_train_filtered.json
# - Xoa video Normal (label == [])
# - Map events_summary tu old sang
# - 52 video khong co trong old: events_summary = ["Fixed"] cho moi event
# - Video 1 event, summary "no anomaly": events_summary = ["Fixed"]
# - Video >= 2 events: drop cac event co summary "no anomaly" (xoa event + summary tuong ung)

In [37]:
filtered = {}
dropped_videos = []

for k, v in train_data.items():
    # Skip normal videos
    if v.get("label", []) == []:
        continue

    entry = dict(v)
    entry.pop("video_summary", None)

    # Video not in old -> mark all events_summary as "Fixed"
    if k not in old_data:
        entry["events_summary"] = ["Fixed"] * len(entry.get("events", []))
        filtered[k] = entry
        continue

    old_entry = old_data[k]
    summaries = old_entry.get("events_summary", [])
    events = entry.get("events", [])

    # 1 event only
    if len(events) == 1:
        if summaries and "no anomaly" in summaries[0].lower():
            entry["events_summary"] = ["Fixed"]
        else:
            entry["events_summary"] = summaries
        filtered[k] = entry
        continue

    # >= 2 events: drop events with "no anomaly" summary
    new_events = []
    new_summaries = []
    for i, (ev, s) in enumerate(zip(events, summaries)):
        if "no anomaly" in s.lower():
            continue
        new_events.append(ev)
        new_summaries.append(s)

    # If no valid events left, drop the video entirely
    if len(new_events) == 0:
        dropped_videos.append(k)
        continue

    entry["events"] = new_events
    entry["events_summary"] = new_summaries
    filtered[k] = entry

with open("ucf_database_train_filtered.json", "w") as f:
    json.dump(filtered, f, indent=2)

print(f"Saved {len(filtered)} anomaly videos to ucf_database_train_filtered.json")

fixed_count = sum(1 for v in filtered.values() if "Fixed" in v.get("events_summary", []))
dropped_events = sum(len(train_data[k].get("events", [])) - len(v.get("events", []))
                     for k, v in filtered.items() if k in old_data)
print(f"Videos with 'Fixed' summary: {fixed_count}")
print(f"Total events dropped (no anomaly): {dropped_events}")
print(f"Videos dropped entirely (all events were no anomaly): {len(dropped_videos)}")
if dropped_videos:
    for v in dropped_videos:
        print(f"  {v}")

Saved 794 anomaly videos to ucf_database_train_filtered.json
Videos with 'Fixed' summary: 77
Total events dropped (no anomaly): 94
Videos dropped entirely (all events were no anomaly): 16
  Abuse024_x264
  Abuse031_x264
  Abuse038_x264
  Abuse041_x264
  Arson034_x264
  Fighting041_x264
  RoadAccidents033_x264
  RoadAccidents118_x264
  Shoplifting014_x264
  Stealing011_x264
  Stealing057_x264
  Stealing077_x264
  Stealing086_x264
  Stealing093_x264
  Vandalism021_x264
  Vandalism041_x264
